<a href="https://colab.research.google.com/github/bojannithya-tech/nithyabojan/blob/main/Assignment_4_RAG_ML_Knowledge_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu transformers sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 89.7 MB/s eta 0:00:00


In [ ]:
import pymupdf
import numpy as np
import pandas as pd
import faiss
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, pipeline

PART 1 — Data Understanding & Preprocessing

Step 1. Upload the PDF

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving intro-to-ml.pdf to intro-to-ml.pdf


In [ ]:
pdf_path = "intro-to-ml.pdf"

Step 2. Load PDF

In [ ]:
doc = pymupdf.open(pdf_path)

print("PDF successfully loaded")
print("Total number of pages:", len(doc))

PDF successfully loaded
Total number of pages: 392


Step 3. Extract text from every page

In [ ]:
pages = []

for page_number in range(len(doc)):
    page = doc.load_page(page_number)
    text = page.get_text("text")

    pages.append({
        "page": page_number + 1,
        "text": text
    })

print("Text extraction completed.")
print("Pages extracted:", len(pages))

Text extraction completed.
Pages extracted: 392


Step 4. Inspect extracted text

In [ ]:
print(pages[14]["text"][:2000])

CHAPTER 1
Introduction
Machine learning is about extracting knowledge from data. It is a research field at the
intersection of statistics, artificial intelligence, and computer science and is also
known as predictive analytics or statistical learning. The application of machine
learning methods has in recent years become ubiquitous in everyday life. From auto‐
matic recommendations of which movies to watch, to what food to order or which
products to buy, to personalized online radio and recognizing your friends in your
photos, many modern websites and devices have machine learning algorithms at their
core. When you look at a complex website like Facebook, Amazon, or Netflix, it is
very likely that every part of the site contains multiple machine learning models.
Outside of commercial applications, machine learning has had a tremendous influ‐
ence on the way data-driven research is done today. The tools introduced in this book
have been applied to diverse scientific problems such as und

Step 5. Check text quality

In [ ]:
page_lengths = [len(p["text"]) for p in pages]

print("Total pages:", len(pages))
print("Pages containing text:",
      sum(length > 0 for length in page_lengths))
print("Empty pages:",
      sum(length == 0 for length in page_lengths))

print("Average characters per page:",
      int(np.mean(page_lengths)))

print("Maximum characters on a page:",
      max(page_lengths))

print("Minimum characters on a page:",
      min(page_lengths))

Total pages: 392
Pages containing text: 387
Empty pages: 5
Average characters per page: 1767
Maximum characters on a page: 4607
Minimum characters on a page: 0


In [ ]:
short_pages = [
    p["page"]
    for p in pages
    if len(p["text"].strip()) < 100
]

print("Pages containing less than 100 characters:")
print(short_pages[:30])

Pages containing less than 100 characters:
[2, 144, 224, 318, 336]


Step 6. Clean the text

In [ ]:
import re

def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

clean_pages = []

for page in pages:
    clean_pages.append({
        "page": page["page"],
        "text": clean_text(page["text"])
    })

print(clean_pages[14]["text"][:1000])

CHAPTER 1 Introduction Machine learning is about extracting knowledge from data. It is a research field at the intersection of statistics, artificial intelligence, and computer science and is also known as predictive analytics or statistical learning. The application of machine learning methods has in recent years become ubiquitous in everyday life. From auto‐ matic recommendations of which movies to watch, to what food to order or which products to buy, to personalized online radio and recognizing your friends in your photos, many modern websites and devices have machine learning algorithms at their core. When you look at a complex website like Facebook, Amazon, or Netflix, it is very likely that every part of the site contains multiple machine learning models. Outside of commercial applications, machine learning has had a tremendous influ‐ ence on the way data-driven research is done today. The tools introduced in this book have been applied to diverse scientific problems such as und

Step 7. Chunk the document

In [ ]:
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(
    embedding_model_name
)


def create_chunks(text, page_number,
                  chunk_size=800,
                  overlap=100):

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):

        end = min(start + chunk_size, len(tokens))

        chunk_tokens = tokens[start:end]

        chunk_text = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append({
            "page": page_number,
            "text": chunk_text
        })

        if end == len(tokens):
            break

        start += chunk_size - overlap

    return chunks

In [ ]:
all_chunks = []

for page in clean_pages:

    page_chunks = create_chunks(
        page["text"],
        page["page"]
    )

    all_chunks.extend(page_chunks)

print("Number of pages:", len(clean_pages))
print("Total chunks generated:", len(all_chunks))
print("Page:", all_chunks[10]["page"])
print()
print(all_chunks[10]["text"][:1500])
chunks_df = pd.DataFrame(all_chunks)

chunks_df.head()
print(chunks_df.shape)

Number of pages: 392
Total chunks generated: 394
Page: 12

this icon indicates a warning or caution. using code examples supplemental material ( code examples, ipython notebooks, etc. ) is available for download at https : / / github. com / amueller / introduction _ to _ ml _ with _ python. this book is here to help you get your job done. in general, if example code is offered with this book, you may use it in your programs and documentation. you do not need to contact us for permission unless you ’ re reproducing a significant portion of the code. for example, writing a program that uses several chunks of code from this book does not require permission. selling or distributing a cd - rom of examples from o ’ reilly books does require permission. answering a question by citing this book and quoting example code does not require permission. incorporating a signifi ‐ cant amount of example code from this book into your product ’ s documentation does require permission. we appreciate, but

PART 2 — Embedding & Vector Database

Step 8. Load SentenceTransformer

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully


Step 9. Generate embeddings

In [ ]:
chunk_texts = chunks_df["text"].tolist()

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding generation completed")
print("Embedding matrix shape:", embeddings.shape)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Embedding generation completed
Embedding matrix shape: (394, 384)


Step 10. Build FAISS vector database

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print("FAISS index created successfully")
print("Vectors stored:", index.ntotal)
print("Embedding dimension:", dimension)

FAISS index created successfully
Vectors stored: 394
Embedding dimension: 384


In [ ]:
normalize_embeddings=True

In [ ]:
faiss.write_index(
    index,
    "ml_book_faiss.index"
)

chunks_df.to_pickle(
    "ml_book_chunks.pkl"
)

print("FAISS database saved")

FAISS database saved


PART 3 — Retrieval Pipeline

Step 11. Retrieval function

In [ ]:
def retrieve_chunks(query, k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        results.append({
            "rank": rank + 1,
            "score": float(scores[0][rank]),
            "page": int(chunks_df.iloc[idx]["page"]),
            "text": chunks_df.iloc[idx]["text"]
        })

    return results

Step 12. Test retrieval

In [ ]:
query = "What is supervised learning?"

results = retrieve_chunks(
    query,
    k=5
)

for result in results:

    print("=" * 80)

    print(
        "Rank:",
        result["rank"]
    )

    print(
        "Similarity Score:",
        round(result["score"], 4)
    )

    print(
        "Page:",
        result["page"]
    )

    print()

    print(
        result["text"][:1000]
    )

Rank: 1
Similarity Score: 0.571
Page: 39

chapter 2 supervised learning as we mentioned earlier, supervised machine learning is one of the most commonly used and successful types of machine learning. in this chapter, we will describe super ‐ vised learning in more detail and explain several popular supervised learning algo ‐ rithms. we already saw an application of supervised machine learning in chapter 1 : classifying iris flowers into several species using physical measurements of the flowers. remember that supervised learning is used whenever we want to predict a certain outcome from a given input, and we have examples of input / output pairs. we build a machine learning model from these input / output pairs, which comprise our training set. our goal is to make accurate predictions for new, never - before - seen data. super ‐ vised learning often requires human effort to build the training set, but afterward automates and often speeds up an otherwise laborious or infeasible task. cl

In [ ]:
results_k3 = retrieve_chunks(
    "What is supervised learning?",
    k=3
)

for r in results_k3:
    print(
        r["rank"],
        r["page"],
        round(r["score"], 4)
    )

1 39 0.571
2 15 0.4838
3 93 0.4557


In [ ]:
results_k5 = retrieve_chunks(
    "What is supervised learning?",
    k=5
)

for r in results_k5:
    print(
        r["rank"],
        r["page"],
        round(r["score"], 4)
    )

1 39 0.571
2 15 0.4838
3 93 0.4557
4 265 0.4297
5 145 0.4254


In [ ]:
results_k7 = retrieve_chunks(
    "What is supervised learning?",
    k=7
)

for r in results_k7:
    print(
        r["rank"],
        r["page"],
        round(r["score"], 4)
    )

1 39 0.571
2 15 0.4838
3 93 0.4557
4 265 0.4297
5 145 0.4254
6 87 0.4246
7 85 0.42


In [ ]:
comparison = pd.DataFrame({
    "Top-K": [3, 5, 7],
    "Number of retrieved chunks": [3, 5, 7],
    "Observation": [
        "Focused context with fewer passages",
        "Good balance between relevance and context",
        "Provides more context but may include less relevant information"
    ]
})

comparison

,Top-K,Number of retrieved chunks,Observation
0,3,3,Focused context with fewer passages
1,5,5,Good balance between relevance and context
2,7,7,Provides more context but may include less rel...


PART 4 — Answer Generation

Step 13. Load the language model

In [6]:
import torch
from transformers import pipeline

print("PyTorch version:", torch.__version__)
print("Pipeline imported successfully")

PyTorch version: 2.11.0+cpu
Pipeline imported successfully


In [7]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=0 if torch.cuda.is_available() else -1
)

print("LLM loaded successfully")


KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [9]:
def rag_assistant(question, k=5):
    """
    End-to-End RAG Pipeline:
    1. Accept user question
    2. Retrieve relevant textbook chunks
    3. Generate grounded answer
    4. Display answer and source information
    """

    print("=" * 80)
    print("MACHINE LEARNING KNOWLEDGE ASSISTANT")
    print("=" * 80)

    print("\nQuestion:")
    print(question)

    # Retrieve relevant chunks and generate answer
    answer, retrieved = generate_answer(question, k=k)

    print("\nAnswer:")
    print(answer)

    print("\nRetrieved Sources:")

    for i, r in enumerate(retrieved, start=1):
        print(
            f"{i}. Page {r['page']} "
            f"| Similarity Score: {r['score']:.4f}"
        )

    print("=" * 80)

    return answer, retrieved

In [10]:
answer, sources = rag_assistant(
    "What is supervised learning?"
)

MACHINE LEARNING KNOWLEDGE ASSISTANT

Question:
What is supervised learning?


NameError: name 'generate_answer' is not defined